# Reliable AI System Evaluation with Cost Optimal Random Sampling

When evaluating an AI system, you often have access to two annotation sources: a cheap but biased LLM-as-judge and an expensive but accurate human annotator. Given a fixed budget, how should you split it between them?

**Cost-Optimal Random Sampling** computes the fraction of samples to send to the expensive annotator so that estimation variance is minimised within your budget.

---

**What you will learn:**

- How to use `CostOptimalRandomSampler` to compute the optimal annotation fraction from a small labeled dataset
- How to use the sampler's outputs with `ASIMeanEstimator` for valid, bias-corrected estimates
- How cost-optimal sampling compares to proxy-only and human-only strategies in balancing cost and accuracy

## The evaluation challenge: two annotation sources, one budget

Suppose you are measuring the hallucination rate of a customer-facing AI assistant. Two annotation sources are available:

- **LLM-as-judge**: $0.05 per sample. Fast and scalable, but systematically under-reports hallucinations.
- **Human annotator**: $10.00 per sample. Ground-truth quality, but 200 times more expensive.

### What you already have

Before the main evaluation, you ran a small pilot study and annotated **100 conversations** with **both** sources. This burn-in dataset serves two purposes:

1. It reveals the judge's bias: the judge systematically under-reports hallucinations compared to human annotators.
2. It allows you to **calibrate** the cost-optimal sampler before spending the main budget.

### The question

You now have **2,000 new conversations** to evaluate and a budget of **$1,500**. Your budget must cover both the proxy cost (`$0.05` per sample) and the human annotation cost (`$10.00` per sample). How do you allocate this budget?

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import numpy as np

from glide.estimators import ASIMeanEstimator, ClassicalMeanEstimator, PPIMeanEstimator
from glide.samplers import CostOptimalRandomSampler
from glide.simulators import generate_binary_dataset

N_BURN_IN = 100
N_MAIN = 2000
N_TOTAL = N_BURN_IN + N_MAIN
TRUE_RATE = 0.10
PROXY_RATE = 0.05
COST_PROXY = 0.05
COST_HUMAN = 10.00
BUDGET = 1500
RANDOM_SEED = 3

C_OPTIMAL = "#27AE60"  # cost-optimal — green
C_PROXY = "#E67E22"  # proxy only   — orange
C_HUMAN = "#2E86AB"  # human only   — blue
C_TRUTH = "#2C3E50"  # true value   — dark slate

plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "#FAFAFA",
        "axes.grid": True,
        "grid.color": "#E5E5E5",
        "grid.linewidth": 0.8,
        "axes.titlesize": 14,
        "axes.titlepad": 14,
        "axes.labelsize": 12,
        "xtick.labelsize": 11,
        "ytick.labelsize": 11,
    }
)

## Simulating 2,100 conversations with a biased judge

`generate_binary_dataset` produces a synthetic dataset that matches the scenario above.

| Array | Meaning |
|---|---|
| `y_true` | Ground-truth binary label: $Y_i = 1$ means a hallucination confirmed by a human annotator |
| `y_proxy` | Binary label from the LLM judge: $\tilde{Y}_i = 1$ means the judge flagged a hallucination |

**Notation key:** Throughout this tutorial, we use code variable names (`y_true`, `y_proxy`, `xi`) in the Python cells and mathematical notation ($Y_i$, $\tilde{Y}_i$, $\xi_i$) in the explanatory text. They correspond as follows: `y_true` ↔ $Y_i$, `y_proxy` ↔ $\tilde{Y}_i$, `xi` ↔ $\xi_i$.

The first `N_BURN_IN = 100` conversations are annotated with **both** sources: these form the burn-in dataset used to calibrate the sampler. The remaining 2,000 conversations are new, and only the proxy label is available initially.

> **Simulated annotation:** In practice, a human label for a new conversation is revealed only after a human annotator's work. Here we generate all ground-truth labels upfront to simulate the annotation process: a main-set conversation only gets its true label revealed when the sampler decides to annotate it (i.e. when $\xi_i = 1$).

In [ ]:
y_true, y_proxy = generate_binary_dataset(
    n_labeled=N_TOTAL,
    n_unlabeled=0,
    true_mean=TRUE_RATE,
    proxy_mean=PROXY_RATE,
    correlation=0.55,
    random_seed=RANDOM_SEED,
)

y_true_burn_in = y_true[:N_BURN_IN]
y_proxy_burn_in = y_proxy[:N_BURN_IN]

In [ ]:
print(f"Total conversations   : {N_TOTAL:,}")
print(f"  Burn-in (labeled)   : {N_BURN_IN}")
print(f"  Main (new)          : {N_MAIN:,}")
print()
print(f"LLM judge cost        : ${COST_PROXY:.2f} per sample")
print(f"Human annotator cost  : ${COST_HUMAN:.2f} per sample")
print(f"Cost ratio            : {COST_HUMAN / COST_PROXY:.0f}x")
print()
print(f"Burn-in: judge rate   = {y_proxy_burn_in.mean():.1%}")
print(f"Burn-in: human rate   = {y_true_burn_in.mean():.1%}")
print(f"Proxy bias            = {y_proxy_burn_in.mean() - y_true_burn_in.mean():+.1%}")

## CostOptimalRandomSampler to the rescue

The burn-in data confirms the proxy is biased, so we cannot trust it directly. We need prediction-powered estimation to correct for this bias. The question is: how many new human annotations should we purchase alongside the cheap proxy labels?

`CostOptimalRandomSampler` answers exactly this question. The workflow has two steps:

1. **Fit** the sampler on the burn-in dataset to estimate the proxy's reliability.
2. **Sample** from the new conversations to obtain annotation assignments.

### Step 1: Fit on the burn-in dataset

The sampler estimates two quantities from the burn-in data:

- $\text{Var}(H)$: variance of the ground-truth labels, capturing how spread out the true outcomes are.
- $\text{MSE}(H, G)$: mean squared error of the proxy relative to ground truth, measuring how unreliable the judge is.

Both quantities are computed during `sampler.fit(y_true, y_proxy)` using the burn-in samples.

In [ ]:
sampler = CostOptimalRandomSampler()
sampler.fit(y_true_burn_in, y_proxy_burn_in)

### The optimal annotation rate

The sampler computes an optimal annotation probability $\pi^*$ based on three factors:

1. **Cost ratio** — How much more expensive is human annotation than the proxy? A 200x cost gap means you need fewer human annotations to stay within budget.
2. **Proxy quality** — How reliable is the LLM judge? A more accurate proxy requires fewer human annotations to correct its bias.
3. **Data variability** — How diverse are the true labels? More variability requires more human annotations to estimate accurately.

The formula for $\pi^*$ balances these trade-offs. In the next step, we call `sampler.sample()` to compute $\pi^*$ and draw Bernoulli indicators for each conversation.

### Step 2: Draw annotation assignments for new conversations

With $\pi^{*}$ determined, the sampler draws independent Bernoulli indicators $\xi_i \sim \text{Bernoulli}(\pi^{*})$ for each new conversation. Conversations with $\xi_i = 1$ are sent for human annotation alongside the LLM label. Conversations with $\xi_i = 0$ receive only the LLM label.

`sampler.sample()` returns two values:

| Return value | Meaning |
|---|---|
| `pi` | The annotation probability $\pi^{*}$ shared by all new conversations |
| `xi` | Bernoulli indicators for each selected sample: $1$ = human annotation obtained, $0$ = proxy label only |

In [ ]:
pi, xi = sampler.sample(
    n_samples=N_MAIN,
    y_true_cost=COST_HUMAN,
    y_proxy_cost=COST_PROXY,
    budget=BUDGET,
    random_seed=RANDOM_SEED,
)

n_human = int(np.nansum(xi))
n_proxy_only = int(np.sum(xi == 0))
n_not_annotated = int(np.sum(np.isnan(xi)))

print(f"Conversations with human annotation   : {n_human}")
print(f"Conversations with proxy-only         : {n_proxy_only}")
print(f"Conversations without annotation      : {n_not_annotated}")
print()
print(f"Annotation probability    : pi = {pi:.3f}")
print(f"Realized annotation rate  : {n_human / (n_human + n_proxy_only):.3f}")

In [ ]:
pi, xi = sampler.sample(
    n_samples=N_MAIN,
    y_true_cost=COST_HUMAN,
    y_proxy_cost=COST_PROXY,
    budget=BUDGET,
    random_seed=RANDOM_SEED,
)
n_human_main = int(np.nansum(xi))
n_processed = np.sum(~np.isnan(xi))

print(f"Conversations processed   : {int(n_processed):,}  (of {N_MAIN:,})")
print(f"Human annotations drawn   : {n_human_main}  ({n_human_main / n_processed:.1%} of processed)")
print(f"Annotation probability    : pi = {pi:.3f}")
print(f"Realized annotation rate  : {n_human_main / n_processed:.3f}")

## Running the full estimation pipeline

`ASIMeanEstimator` implements **Active Statistical Inference (ASI)**, which combines human and proxy labels to produce unbiased, variance-reduced estimates:

- **For human-annotated samples** ($\xi_i = 1$): Use inverse probability weighting (IPW) to correct for selective sampling bias caused by the low annotation rate $\pi^*$.
- **For proxy-only samples** ($\xi_i = 0$): Use the proxy label, weighted by the estimator's knowledge of the proxy's reliability.
- **For burn-in samples** ($\pi = 1$): Use the full human label since all were annotated.

To use `ASIMeanEstimator`, build arrays of length `N_TOTAL` containing:

1. `y_true_full`: Human labels for annotated samples, NaN elsewhere.
2. `pi_full`: The annotation probability for each sample (1.0 for burn-in, $\pi^*$ for main).
3. Pass both arrays plus `y_proxy` to `ASIMeanEstimator().estimate()`.

In [ ]:
y_true_full = np.full(N_TOTAL, np.nan)
y_true_full[:N_BURN_IN] = y_true_burn_in

annotated_main_indices = np.where(xi == 1)[0]
y_true_full[N_BURN_IN + annotated_main_indices] = y_true[N_BURN_IN + annotated_main_indices]

pi_full = np.full(N_TOTAL, pi)
pi_full[:N_BURN_IN] = 1.0

result_demo = ASIMeanEstimator().estimate(
    y_true_full,
    y_proxy,
    pi_full,
    metric_name="Hallucination Rate",
    confidence_level=0.95,
)

In [ ]:
n_human_total_demo = N_BURN_IN + n_human_main
print(result_demo.summary())
print()
print(f"Human labels used : {n_human_total_demo}  ({N_BURN_IN} burn-in + {n_human_main} main)")
print(f"Proxy labels used : {N_TOTAL}")
print(f"True rate         : {TRUE_RATE:.1%}")

## Comparing with two baselines at a fixed budget

You already have the burn-in dataset. Now, with your `$1,500` budget to evaluate the 2,000 new conversations, how should you allocate it? We compare three strategies:

**Proxy-only baseline:** Ignore the burn-in's signal about bias. Buy cheap proxy labels for all 2,000 new conversations; spend the remaining budget on nothing. Use a PPI estimator with only the burn-in's 100 human labels as ground truth. The new signal comes entirely from proxy labels, so bias correction is weak and estimation quality is limited by proxy accuracy alone.

**Human-only baseline:** Ignore the burn-in as a calibration tool. Spend your entire `$1,500` budget on human annotations: at `$10.05` per sample (proxy + human), this buys 149 new human labels. Use a classical estimator on burn-in (100) plus new (149) samples. You ignore all proxy labels, forgoing the variance reduction available from multi-source estimation.

**Cost-optimal:** Use the burn-in to calibrate the sampler and learn the proxy's reliability. Compute the optimal annotation rate π*, then allocate your budget across 2,000 proxy labels and selectively chosen human annotations. ASI with IPW corrects for selective sampling and leverages both signals for efficient variance reduction.

In [ ]:
# Proxy-only: PPIMeanEstimator with burn-in as labeled, all main as unlabeled proxy
y_true_proxy_only = np.full(N_TOTAL, np.nan)
y_true_proxy_only[:N_BURN_IN] = y_true_burn_in
result_proxy_only = PPIMeanEstimator().estimate(
    y_true_proxy_only,
    y_proxy,
    metric_name="Hallucination Rate",
)

# Human-only: ClassicalMeanEstimator on burn-in + as many main samples as budget allows
n_human_only_main = int(np.floor(BUDGET / (COST_PROXY + COST_HUMAN)))
rng = np.random.default_rng(RANDOM_SEED + 1)
human_only_main_idx = np.sort(rng.choice(N_MAIN, size=n_human_only_main, replace=False))
y_true_human_only = np.hstack(
    [
        y_true_burn_in,
        y_true[N_BURN_IN + human_only_main_idx],
    ]
)
result_human_only = ClassicalMeanEstimator().estimate(
    y_true_human_only,
    metric_name="Hallucination Rate",
)

n_human_total_human_only = N_BURN_IN + n_human_only_main
print(f"Proxy-only  : {N_BURN_IN} human labels + {N_TOTAL} proxy labels")
print(f"Human-only  : {n_human_total_human_only} human labels total  ({n_human_only_main} new at ${BUDGET})")
print(f"Cost-optimal: {n_human_total_demo} human labels + {N_TOTAL} proxy labels  (pi* = {pi:.2f})")

In [ ]:
baseline_estimates = [
    (
        f"Proxy-only\n({N_BURN_IN} human  +  {N_TOTAL:,} proxy)\n"
        f"(no new human annotations, aside from burn-in dataset)",
        result_proxy_only.mean,
        result_proxy_only.confidence_interval.lower_bound,
        result_proxy_only.confidence_interval.upper_bound,
        C_PROXY,
    ),
    (
        f"Human-only\n({n_human_total_human_only} human  |  no PPI)\n(${BUDGET} budget)",
        result_human_only.mean,
        result_human_only.confidence_interval.lower_bound,
        result_human_only.confidence_interval.upper_bound,
        C_HUMAN,
    ),
    (
        f"Cost-optimal (GLIDE)\n({n_human_total_demo} human  +  {N_TOTAL:,} proxy)\n(pi* = {pi:.2f}, ${BUDGET} budget)",
        result_demo.mean,
        result_demo.confidence_interval.lower_bound,
        result_demo.confidence_interval.upper_bound,
        C_OPTIMAL,
    ),
]

fig, ax = plt.subplots(figsize=(11, 5.5))
y_pos = [2, 1, 0]

for y, (label, mean, lo, hi, color) in zip(y_pos, baseline_estimates):
    ax.plot([lo, hi], [y, y], color=color, linewidth=4, solid_capstyle="round", zorder=3)
    for xc in [lo, hi]:
        ax.plot([xc, xc], [y - 0.2, y + 0.2], color=color, linewidth=2.5, zorder=3)
    ax.scatter(mean, y, s=200, color=color, zorder=5, edgecolors="white", linewidths=2.5)
    ax.text(mean, y + 0.34, f"{mean:.1%}", ha="center", va="bottom", fontsize=12, color=color, fontweight="bold")
    ax.text(mean, y - 0.34, f"[{lo:.1%},  {hi:.1%}]", ha="center", va="top", fontsize=11, color="#888888")

ax.axvline(TRUE_RATE, color=C_TRUTH, linestyle="--", linewidth=2.5, zorder=4)
ax.text(TRUE_RATE + 0.004, 2.72, "True rate  10%", color=C_TRUTH, fontsize=10.5, fontweight="bold")

ax.set_yticks(y_pos)
ax.set_yticklabels([e[0] for e in baseline_estimates], fontsize=11)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
ax.set_xlabel("Estimated Hallucination Rate", fontsize=12)
ax.set_title(
    "Comparing Cost-Optimal, Proxy-Only, and Human-Only Strategies",
    fontsize=14,
    fontweight="bold",
)
ax.set_xlim(-0.01, 0.26)
ax.set_ylim(-0.8, 3.2)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(left=False)
plt.tight_layout()
plt.show()

## Summary: what each strategy contributes

| | Proxy-only | Human-only | **Cost-optimal** |
|---|---|---|---|
| New proxy labels | 2,000 | 149 | 2,000 |
| New human labels | 0 | 149 | ~136 |
| Annotation probability | n/a | 1 | 0.088 |
| Unbiased estimate | limited by burn-in | yes | yes |
| Leverages all proxy labels | yes (via PPI) | no | yes (via ASI + IPW) |
| Confidence interval narrows with more budget | no | yes (slowly) | yes (efficiently) |

**Key takeaways:**

1. **The burn-in dataset does double duty.** It calibrates the sampler and contributes labeled samples to the final estimate. No data is wasted.

2. **π* is determined by the cost ratio and proxy quality.** A 200x cost gap and a moderately accurate proxy yield π* around 0.088: roughly one in eleven new conversations needs a human label to achieve near-optimal precision.

3. **Proxy-only is limited by burn-in.** Spending your remaining budget entirely on cheap proxy labels gives you many more samples but adds no new ground-truth signal beyond what the burn-in already provides. The confidence interval does not shrink.

4. **Human-only wastes the proxy labels.** Spending your budget on full annotations ignores the variance reduction available from leveraging all proxy labels. You get fewer human labels per dollar and no PPI benefit.

5. **Cost-optimal gets the best of both worlds.** It covers all new conversations with proxy labels and spends just enough on human annotations to correct the bias. The same budget yields tighter confidence intervals than either alternative.

---

*Want to go further? The [ASI tutorial](../asi/) shows how to concentrate human annotations on the most uncertain conversations, adding an active layer on top of cost optimisation.*